In [1]:
from pyspark.sql import SparkSession

# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("MAST30034 Tutorial 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/18 17:49:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/18 17:49:54 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
import pandas as pd

# data output directory is
output_relative_dir = '../../data/'
abs_output_dir = output_relative_dir + 'raw_abs'

# Reading in the POA <-> SA2 mapping dataset produced in the previous run 
poa_to_sa2 = pd.read_parquet(f"{abs_output_dir}/poa_to_sa2.parquet")
print(poa_to_sa2.info())
print(poa_to_sa2.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2644 entries, 0 to 2643
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   POA_CODE_2021  2644 non-null   object 
 1   SA2_CODE_2021  2644 non-null   object 
 2   mb_count       2644 non-null   int64  
 3   poa_total      2644 non-null   int64  
 4   ratio          2644 non-null   float64
dtypes: float64(1), int64(2), object(2)
memory usage: 103.4+ KB
None
  POA_CODE_2021 SA2_CODE_2021  mb_count  poa_total     ratio
0          0800     701011002        93         93  1.000000
1          0810     701021025        69        459  0.150327
2          0812     701021022        74        266  0.278195
3          0820     701011008        78        365  0.213699
4          0822     702041063       131        511  0.256360


In [3]:
poa_to_sa2['POA_CODE_2021'].duplicated().sum()

np.int64(0)

In [4]:
# Reading in the actual tbl consumer data 
tbl_consumer_df = pd.read_csv("../../data/tables/part_1/tbl_consumer.csv", sep="|")
print(tbl_consumer_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 499999 entries, 0 to 499998
Data columns (total 6 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   name         499999 non-null  object
 1   address      499999 non-null  object
 2   state        499999 non-null  object
 3   postcode     499999 non-null  int64 
 4   gender       499999 non-null  object
 5   consumer_id  499999 non-null  int64 
dtypes: int64(2), object(4)
memory usage: 22.9+ MB
None


Since the "postcode" column in tbl_consumer_df is stored as an integer and the poa_to_sa2 stores "POA_CODE_2021" as a string of four characters, we need to make sure their data typesa are consistent. Therefore, postcode columns data type will be adjusted accordingly. 

In [5]:
tbl_consumer_df['postcode'] = tbl_consumer_df['postcode'].astype(str).str.zfill(4)

In [6]:
# Merging consumer data to the SA2 data via postcode
consumer_sa2_df = tbl_consumer_df.merge(
    poa_to_sa2, 
    left_on="postcode", 
    right_on="POA_CODE_2021", 
    how= "left" )
print(consumer_sa2_df.info)
print(consumer_sa2_df.shape)


<bound method DataFrame.info of                      name                          address state postcode  \
0        Yolanda Williams       413 Haney Gardens Apt. 742    WA     6935   
1              Mary Smith                  3764 Amber Oval   NSW     2782   
2           Jill Jones MD               40693 Henry Greens    NT     0862   
3         Lindsay Jimenez        00653 Davenport Crossroad   NSW     2780   
4       Rebecca Blanchard    9271 Michael Manors Suite 651    WA     6355   
...                   ...                              ...   ...      ...   
499994      Jessica Avila    508 Miranda Overpass Apt. 218   QLD     4400   
499995    Steven Thornton  7913 Schwartz Mission Suite 483   VIC     3097   
499996      Christy Smith   5681 Zachary Mountain Apt. 060   NSW     2756   
499997       Donna Sutton                54140 Jacob Point   VIC     3989   
499998     Hannah Wilkins                61055 Long Valley   NSW     1755   

             gender  consumer_id POA_CODE_2

In [7]:
# Checking for missing values 
poa_to_sa2['POA_CODE_2021'].duplicated().sum()
poa_to_sa2 = poa_to_sa2.drop_duplicates(subset='POA_CODE_2021', keep='first')

In [8]:
# Checking for missing SA2 values where it is NaN
consumer_sa2_df['SA2_CODE_2021'].isna().sum()

np.int64(83181)

In [9]:
# Identifying the reason for the NaN SA2 values that might have occured during the merging due to missing matching postcodes 
consumer_postcodes = set(tbl_consumer_df['postcode'].unique())
poa_postcodes = set(poa_to_sa2['POA_CODE_2021'].unique())

missing_postcodes = consumer_postcodes - poa_postcodes
len(missing_postcodes)
list(missing_postcodes)[:20]

['5942',
 '1187',
 '1295',
 '1030',
 '1670',
 '3353',
 '6865',
 '3643',
 '6989',
 '8006',
 '1163',
 '3800',
 '6913',
 '1141',
 '6985',
 '1029',
 '2013',
 '2004',
 '1239',
 '1570']

Inspecting the missing SA2 values during the merge shows 83,181 rows with NaN SA2 value, this is around 16.6% (83,181/499,999 of total). After closely investigating and researching, it was found that these postcodes are reserved for Australia's non geographic post code ranges exclusively. The ABS Postal Area is built from mesh blocks but since these codes do not have a geographic location linekd to it, there is no SA2 to allocate it to. 

In [12]:
# data output directory is
output_relative_dir = '../../data/'
abs_output_dir = output_relative_dir + 'raw_abs'

# save the cleaned POA <-> SA2 mapping df to be reused in the next step 
consumer_sa2_df.to_parquet(f"{abs_output_dir}/consumer_sa2.parquet", index=False)
